In [0]:
# Databricks notebook source
# NYC Taxi Fare Prediction - End-to-End PySpark (EDA + Feature Eng + ML + Tuning)
# Dataset columns:
# key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count

from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
BASE_DIR = "/Volumes/workspace/default/nyc/"
DATA_PATH = BASE_DIR + "train_part_0.csv"   # GANTI kalo nama file beda

schema = T.StructType([
    T.StructField("key", T.StringType(), True),
    T.StructField("fare_amount", T.DoubleType(), True),
    T.StructField("pickup_datetime", T.StringType(), True),
    T.StructField("pickup_longitude", T.DoubleType(), True),
    T.StructField("pickup_latitude", T.DoubleType(), True),
    T.StructField("dropoff_longitude", T.DoubleType(), True),
    T.StructField("dropoff_latitude", T.DoubleType(), True),
    T.StructField("passenger_count", T.IntegerType(), True),
])

In [0]:
# Check if the file path exists first
try:
    files = dbutils.fs.ls(BASE_DIR)
    print(f"Files in {BASE_DIR}:")
    for f in files:
        print(f"  {f.name}")
except Exception as e:
    print(f"Error listing directory: {e}")
    print(f"\nThe volume path {BASE_DIR} may not exist.")
    print("Please verify:")
    print("  1. The catalog 'workspace' exists")
    print("  2. The schema 'default' exists")
    print("  3. The volume 'nyc' exists")
    raise

df_raw = (spark.read
          .option("header", True)
          .schema(schema)
          .csv(DATA_PATH))

df_raw.printSchema()
display(df_raw.limit(5))

In [0]:
cols = df_raw.columns

missing = df_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in cols
])
display(missing)

In [0]:
display(df_raw.select(
    "fare_amount", "passenger_count",
    "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude"
).describe())

In [0]:
q_fare = df_raw.approxQuantile("fare_amount", [0.25, 0.5, 0.75], 0.01)
q_pass = df_raw.approxQuantile("passenger_count", [0.25, 0.5, 0.75], 0.01)

print("Fare Q1, Median, Q3:", q_fare)
print("Passenger Q1, Median, Q3:", q_pass)

In [0]:
# Contoh input: "2009-06-15 17:26:21 UTC"
df1 = (df_raw
       .withColumn("pickup_ts", F.to_timestamp("pickup_datetime", "yyyy-MM-dd HH:mm:ss 'UTC'")))

# Quick check hasil parse
display(df1.select("pickup_datetime", "pickup_ts").limit(10))

# Fitur waktu
df1 = (df1
       .withColumn("pickup_hour", F.hour("pickup_ts"))
       .withColumn("pickup_dow", F.dayofweek("pickup_ts"))
       .withColumn("pickup_month", F.month("pickup_ts"))
       .withColumn("pickup_year", F.year("pickup_ts")))

# Kalau pickup_ts masih banuak null
# display(df_raw.select("pickup_datetime").where(F.col("pickup_datetime").isNotNull()).limit(20))

In [0]:
# Rule dasar
df2 = df1.filter(
    (F.col("fare_amount").isNotNull()) &
    (F.col("fare_amount") > 0) &
    (F.col("passenger_count").between(1, 6)) &
    (F.col("pickup_ts").isNotNull())
)

# Bounding box NYC (bisa kamu sesuaikan)
df2 = df2.filter(
    F.col("pickup_longitude").between(-74.2591, -73.7004) &
    F.col("dropoff_longitude").between(-74.2591, -73.7004) &
    F.col("pickup_latitude").between(40.4774, 40.9176) &
    F.col("dropoff_latitude").between(40.4774, 40.9176)
)

display(df2.limit(5))

In [0]:
R = 6371.0  # km

df3 = (df2
       .withColumn("pickup_coord", F.struct(F.col("pickup_latitude").alias("lat"),
        F.col("pickup_longitude").alias("lon")))
       .withColumn("dropoff_coord", F.struct(F.col("dropoff_latitude").alias("lat"),
        F.col("dropoff_longitude").alias("lon")))
      )

df3 = (df3
    .withColumn("pickup_lat_rad", F.radians(F.col("pickup_latitude")))
    .withColumn("pickup_lon_rad", F.radians(F.col("pickup_longitude")))
    .withColumn("dropoff_lat_rad", F.radians(F.col("dropoff_latitude")))
    .withColumn("dropoff_lon_rad", F.radians(F.col("dropoff_longitude")))
    .withColumn("dlat", F.col("dropoff_lat_rad") - F.col("pickup_lat_rad"))
    .withColumn("dlon", F.col("dropoff_lon_rad") - F.col("pickup_lon_rad"))
    .withColumn("a", (F.sin(F.col("dlat")/2) ** 2) +
                (F.cos(F.col("pickup_lat_rad")) * F.cos(F.col("dropoff_lat_rad")) *
                (F.sin(F.col("dlon")/2) ** 2)))
    .withColumn("c", F.lit(2.0) * F.asin(F.sqrt(F.col("a"))))
    .withColumn("distance_km", F.lit(R) * F.col("c"))
)

# Filter jarak masuk akal (atur sesuai kebutuhan)
df3 = df3.filter((F.col("distance_km") > 0) & (F.col("distance_km") <= 1000))

# Drop kolom intermediate biar ringan
drop_cols = ["pickup_lat_rad","pickup_lon_rad","dropoff_lat_rad","dropoff_lon_rad","dlat","dlon","a","c"]
df = df3.drop(*drop_cols)

display(df.select("fare_amount","distance_km","passenger_count","pickup_hour","pickup_coord","dropoff_coord").limit(5))

In [0]:
display(df.select("fare_amount","distance_km","passenger_count","pickup_hour","pickup_dow","pickup_month").describe())

In [0]:
q_dist = df.approxQuantile("distance_km", [0.25, 0.5, 0.75, 0.95, 0.99], 0.01)
q_fare2 = df.approxQuantile("fare_amount", [0.25, 0.5, 0.75, 0.95, 0.99], 0.01)

print("Distance quantiles:", q_dist)
print("Fare quantiles:", q_fare2)

In [0]:
display(df.selectExpr(
    "corr(distance_km, fare_amount) as corr_dist_fare",
    "corr(passenger_count, fare_amount) as corr_pass_fare",
    "corr(distance_km, passenger_count) as corr_dist_pass"
))

In [0]:
pass_dist = (df.groupBy("passenger_count")
             .count()
             .orderBy("passenger_count"))

total = df.count()
pass_dist = pass_dist.withColumn("proportion", F.col("count") / F.lit(total))
display(pass_dist)

In [0]:
df_bucket = (df
    .withColumn(
        "distance_bucket",
        F.when(F.col("distance_km") < 2, "0-2 km")
         .when(F.col("distance_km") < 5, "2-5 km")
         .when(F.col("distance_km") < 10, "5-10 km")
         .when(F.col("distance_km") < 20, "10-20 km")
         .otherwise(">20 km")
    )
)

bucket_stats = (df_bucket.groupBy("distance_bucket")
    .agg(
        F.count("*").alias("trips"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("distance_km").alias("avg_distance"),
        F.expr("percentile_approx(fare_amount, 0.5)").alias("median_fare")
    )
    .orderBy("distance_bucket")
)

display(bucket_stats)

In [0]:
by_hour = (df.groupBy("pickup_hour")
           .agg(
               F.count("*").alias("trips"),
               F.avg("fare_amount").alias("avg_fare"),
               F.avg("distance_km").alias("avg_distance")
           )
           .orderBy("pickup_hour"))

display(by_hour)

In [0]:
# Ambil sample kecil untuk plotting (sesuaikan fraction)
sample_frac = 0.001  # 0.1% ~ 55k rows dari 55 juta (perkiraan)
df_s = df.sample(withReplacement=False, fraction=sample_frac, seed=42).select(
    "fare_amount","distance_km","passenger_count","pickup_hour","pickup_dow","pickup_month"
)

pdf = df_s.toPandas()
pdf.shape

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_style("whitegrid")

# 1) Histogram
fig, ax = plt.subplots(1, 2, figsize=(14,4))
sns.histplot(pdf["fare_amount"], bins=100, ax=ax[0])
ax[0].set_title("Histogram Fare Amount")
sns.histplot(pdf["distance_km"], bins=100, ax=ax[1])
ax[1].set_title("Histogram Distance (km)")
plt.show()

# 2) Boxplot (outlier)
fig, ax = plt.subplots(1, 2, figsize=(14,4))
sns.boxplot(x=pdf["fare_amount"], ax=ax[0])
ax[0].set_title("Boxplot Fare Amount")
sns.boxplot(x=pdf["distance_km"], ax=ax[1])
ax[1].set_title("Boxplot Distance (km)")
plt.show()

# 3) Scatter: distance vs fare (warna passenger_count)
plt.figure(figsize=(7,5))
sns.scatterplot(data=pdf.sample(min(len(pdf), 20000), random_state=1),
                x="distance_km", y="fare_amount", hue="passenger_count", alpha=0.4, s=12)
plt.title("Scatter: Distance vs Fare (colored by Passenger Count)")
plt.show()

# 4) Heatmap korelasi
corr = pdf[["fare_amount","distance_km","passenger_count","pickup_hour","pickup_dow","pickup_month"]].corr()
plt.figure(figsize=(7,5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

# 5) Q-Q Plot (fare)
plt.figure(figsize=(6,6))
stats.probplot(pdf["fare_amount"].dropna(), dist="norm", plot=plt)
plt.title("Q-Q Plot: Fare Amount vs Normal")
plt.show()

# 6) Violin plot: fare by hour (ambil subset biar tidak berat)
plt.figure(figsize=(14,5))
subset = pdf[pdf["pickup_hour"].notna()].copy()
subset = subset.sample(min(len(subset), 30000), random_state=2)
sns.violinplot(data=subset, x="pickup_hour", y="fare_amount", inner="quartile", cut=0)
plt.title("Violin Plot: Fare by Pickup Hour")
plt.show()

# 7) ECDF: fare
x = np.sort(pdf["fare_amount"].dropna().values)
y = np.arange(1, len(x)+1) / len(x)
plt.figure(figsize=(7,5))
plt.plot(x, y)
plt.title("ECDF: Fare Amount")
plt.xlabel("Fare Amount")
plt.ylabel("ECDF")
plt.show()

# (Opsional) Pair plot (berat, gunakan sample kecil)
pp = pdf[["fare_amount","distance_km","passenger_count","pickup_hour"]].dropna().sample(5000, random_state=3)
sns.pairplot(pp, corner=True)
plt.show()

In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA

feature_cols = [
    "pickup_longitude","pickup_latitude",
    "dropoff_longitude","dropoff_latitude",
    "distance_km",
    "passenger_count",
    "pickup_hour","pickup_dow","pickup_month"
]

df_ml = df.dropna(subset=feature_cols + ["fare_amount"])

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_vec = assembler.transform(df_ml)

scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withStd=True, withMean=True)
scaler_model = scaler.fit(df_vec)
df_scaled = scaler_model.transform(df_vec)

pca = PCA(k=5, inputCol="scaled_features", outputCol="pca_features")
pca_model = pca.fit(df_scaled)

print("Explained variance (k=5):", pca_model.explainedVariance)

In [0]:
train, test = df_vec.randomSplit([0.8, 0.2], seed=42)

In [0]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

lr = LinearRegression(featuresCol="features", labelCol="fare_amount")

lr_model = lr.fit(train)
pred_lr = lr_model.transform(test)

e_rmse = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="rmse")
e_mae  = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="mae")
e_r2   = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="r2")

print("LR RMSE:", e_rmse.evaluate(pred_lr))
print("LR MAE :", e_mae.evaluate(pred_lr))
print("LR R2  :", e_r2.evaluate(pred_lr))

In [0]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="fare_amount",
    maxIter=50,
    maxDepth=5,
    stepSize=0.1,
    subsamplingRate=0.8
)

gbt_model = gbt.fit(train)
pred_gbt = gbt_model.transform(test)

print("GBT RMSE:", e_rmse.evaluate(pred_gbt))
print("GBT MAE :", e_mae.evaluate(pred_gbt))
print("GBT R2  :", e_r2.evaluate(pred_gbt))

In [0]:
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

gbt = GBTRegressor(featuresCol="features", labelCol="fare_amount")

paramGrid = (ParamGridBuilder()
    .addGrid(gbt.maxDepth, [3, 5, 7])
    .addGrid(gbt.maxIter, [30, 50])
    .addGrid(gbt.stepSize, [0.05, 0.1])
    .addGrid(gbt.subsamplingRate, [0.7, 0.9])
    .build())

tvs = TrainValidationSplit(
    estimator=gbt,
    estimatorParamMaps=paramGrid,
    evaluator=RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="rmse"),
    trainRatio=0.8,
    parallelism=4
)

tvs_model = tvs.fit(train)
pred_tuned = tvs_model.transform(test)

print("Tuned GBT RMSE:", e_rmse.evaluate(pred_tuned))
print("Tuned GBT MAE :", e_mae.evaluate(pred_tuned))
print("Tuned GBT R2  :", e_r2.evaluate(pred_tuned))

best_model = tvs_model.bestModel
print("Best maxDepth:", best_model.getMaxDepth())
print("Best maxIter :", best_model.getMaxIter())
print("Best stepSize:", best_model.getStepSize())
print("Best subsamplingRate:", best_model.getSubsamplingRate())